# grid-rbd control surface — plant step, costs & barriers

The `grid_plant` surface exposed on the handle gives a DDP/iLQR/SQP step its
building blocks: `plant_step` (the integrator), quadratic state/input costs
(value + grad + Gauss-Newton Hessian), an end-effector position cost
(`Jₚᵀ W Jₚ`), and log-barriers for joint/velocity/torque limits. We walk one
toy control step on iiwa14 and validate each piece against a closed-form
recompute.

Requires: CUDA GPU + `nvcc` + `grid_rbd`. iiwa14 = seconds to compile.

In [ ]:
import numpy as np
from pathlib import Path
import grid_rbd
URDF = next(p / 'robot_assets' / 'iiwa14.urdf' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'robot_assets' / 'iiwa14.urdf').exists())
np.random.seed(0)
h = grid_rbd.register_robot('iiwa14_plant_nb', urdf_path=str(URDF),
                            floating_base=False, max_batch_size=16)
NJ, NV = h.num_joints, h.num_vel
NX = NJ + NV   # state x = [q; qd]
print(h, ' NX =', NX)

## 1. plant_step — one integrator step x_{k+1} = f(x_k, u_k, dt)

In [ ]:
B = 4
x = np.random.randn(B, NX).astype(np.float32) * 0.1
u = np.random.randn(B, NV).astype(np.float32) * 0.1
dt = 0.01
x_next = h.plant_step(x, u, dt, integrator_type='euler')
print('x_next:', x_next.shape)
assert x_next.shape == (B, NX)
# euler: q_{k+1} = q_k + dt*qd_k for the position block
q_next_expected = x[:, :NJ] + dt * x[:, NJ:NJ+NV]
assert np.max(np.abs(x_next[:, :NJ] - q_next_expected)) < 5e-3

## 2. Quadratic state & input costs (value, grad, GN Hessian)

`quadratic_state_cost(x, x_des, Q) = ½ Σ Qᵢ (xᵢ−x_desᵢ)²`. Grad is
`Q⊙(x−x_des)`; the Gauss-Newton Hessian is `diag(Q)`.

In [ ]:
x_des = np.zeros((B, NX), dtype=np.float32)
Q = (np.random.rand(B, NX).astype(np.float32) + 0.5)
val, grad, hess = h.quadratic_state_cost(x, x_des, Q)
val_ref  = 0.5 * np.sum(Q * (x - x_des)**2, axis=1)
grad_ref = Q * (x - x_des)
assert np.max(np.abs(val - val_ref)) < 5e-3
assert np.max(np.abs(grad - grad_ref)) < 5e-3
# GN Hessian diagonal == Q
diag = np.stack([np.diag(hess[b]) for b in range(B)])
assert np.max(np.abs(diag - Q)) < 5e-3
print('state cost: value/grad/diag(Hessian) all match diag(Q)')

R = (np.random.rand(B, NV).astype(np.float32) + 0.5)
u_des = np.zeros((B, NV), dtype=np.float32)
uval, ugrad, uhess = h.quadratic_input_cost(u, u_des, R)
assert np.max(np.abs(uval - 0.5*np.sum(R*(u-u_des)**2, axis=1))) < 5e-3
print('input cost matches diag(R) too')

## 3. End-effector position cost — value + GN Hessian `Jₚᵀ W Jₚ`

In [ ]:
q = np.random.randn(B, NJ).astype(np.float32) * 0.2
p_des = np.zeros((B, 3), dtype=np.float32)
W = np.ones((B, 3), dtype=np.float32)
eval_, egrad, ehess = h.ee_pos_cost(q, p_des, W)
# closed-form: r = p(q) - p_des; value = 1/2 r^T W r
p = h.end_effector_pose(q)[:, :3]
r = p - p_des
val_ref = 0.5 * np.sum(W * r**2, axis=1)
assert np.max(np.abs(eval_ - val_ref)) < 5e-3
# GN Hessian top-left NV-block = Jp^T diag(W) Jp (symmetric PSD)
J = h.end_effector_pose_gradient(q)[:, :3, :]    # linear (xyz) rows (B,3,NV)
for b in range(B):
    H_ref = J[b].T @ np.diag(W[b]) @ J[b]
    H = ehess[b][:NV, :NV]
    assert np.max(np.abs(H - H_ref)) < 5e-3
print('ee_pos_cost value + GN Hessian match Jp^T W Jp')

## 4. Log-barriers for joint limits (unbounded DOF ⇒ exactly zero)

In [ ]:
lo = -1.5 * np.ones((B, NJ), dtype=np.float32)
hi =  1.5 * np.ones((B, NJ), dtype=np.float32)
var = np.clip(q, -1.0, 1.0).astype(np.float32)
mu = 0.1
bval, bgrad, bhess = h.joint_position_barrier(var, lo, hi, mu)
# closed form: b = -mu*(log(x-lo) + log(hi-x)) summed over DOF
b_ref = -mu * np.sum(np.log(var - lo) + np.log(hi - var), axis=1)
assert np.max(np.abs(bval - b_ref)) < 5e-3
# an inf bound contributes exactly zero
lo_inf = np.full((B, NJ), -np.inf, dtype=np.float32)
hi_inf = np.full((B, NJ),  np.inf, dtype=np.float32)
bval0, bgrad0, bhess0 = h.joint_position_barrier(var, lo_inf, hi_inf, mu)
assert np.max(np.abs(bval0)) == 0.0
assert np.max(np.abs(bgrad0)) == 0.0
print('barrier matches log-barrier; unbounded DOF -> exactly zero')

All four `grid_plant` pieces (`plant_step`, quadratic costs, ee-position
cost, log-barrier) validated — these are exactly what one Gauss-Newton
trajectory-optimization step needs, batched over candidate trajectories.